# Study Context

In [11]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from study_context import ExperimentContext, pipeline_for_accession_list

load_dotenv()

ROOT = Path().resolve().parents[0]

## Config

In [12]:
# --- config ---
READ_FROM_CACHE = False
SAMPLE_SIZE = None  # set to int to limit accessions for testing, e.g. SAMPLE_SIZE = 5

ACCESSIONS_PATH = ROOT / "output/metadata/datasets.csv"
OUTPUT_PATH = ROOT / "output/context/contexts.jsonl"

## Fetch / Load

In [13]:
if READ_FROM_CACHE and OUTPUT_PATH.exists():
    with open(OUTPUT_PATH) as f:
        contexts = [ExperimentContext.model_validate_json(line) for line in f if line.strip()]
    print(f"Loaded {len(contexts)} contexts from cache.")
else:
    data = pd.read_csv(ACCESSIONS_PATH)
    if SAMPLE_SIZE:
        data = data.sample(n=SAMPLE_SIZE, random_state=42)
    accessions = data["srx_accession"].tolist()
    print(f"Fetching context for {len(accessions)} accessions...")
    contexts = pipeline_for_accession_list(accessions)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_PATH, "w") as f:
        for ctx in contexts:
            f.write(ctx.model_dump_json() + "\n")
    print(f"Saved {len(contexts)} contexts to {OUTPUT_PATH}")

Fetching context for 801 accessions...
[2026-04-21 16:26:12] Fetching experiment context for accession: ERX11662371
[2026-04-21 16:26:12] Fetching experiment context for accession: SRX19233503
[2026-04-21 16:26:12] Fetching experiment context for accession: ERX11662373
[2026-04-21 16:26:12] Fetching experiment context for accession: ERX11662385
[2026-04-21 16:26:12] Fetching experiment context for accession: ERX11662357
[2026-04-21 16:26:12] Fetching experiment context for accession: ERX11662360
[2026-04-21 16:26:12] Fetching experiment context for accession: ERX11662364
[2026-04-21 16:26:12] Fetching experiment context for accession: ERX11662370
[2026-04-21 16:26:13] Fetching PubMed abstracts for PMIDs: ['36803741']
[2026-04-21 16:26:13] Fetching experiment context for accession: ERX11662369
[2026-04-21 16:26:13] Fetching experiment context for accession: ERX11662367
[2026-04-21 16:26:13] Fetching experiment context for accession: ERX11662377
[2026-04-21 16:26:13] Fetching experiment 

## Basic Counts

In [14]:
# basic counts: check for missing, extra, and duplicate accessions
source_accessions = pd.read_csv(ACCESSIONS_PATH)["srx_accession"].tolist()
loaded_accessions = [ctx.accession for ctx in contexts]

missing = set(source_accessions) - set(loaded_accessions)
extra   = set(loaded_accessions) - set(source_accessions)
dupes   = [a for a in loaded_accessions if loaded_accessions.count(a) > 1]

print(f"Source accessions:     {len(source_accessions)}")
print(f"Loaded contexts:       {len(contexts)}")
print(f"Missing from contexts: {len(missing)}")
print(f"Extra in contexts:     {len(extra)}")
print(f"Duplicates:            {len(set(dupes))}")

if missing:
    print("\nMissing accessions:", sorted(missing))

Source accessions:     801
Loaded contexts:       801
Missing from contexts: 0
Extra in contexts:     0
Duplicates:            0


## Field Coverage

In [15]:
# field coverage
n = len(contexts)

fields = {
    "studyDescription":                      sum(1 for c in contexts if c.study and c.study.studyDescription),
    "pubmedAbstract":                        sum(1 for c in contexts if c.study and c.study.pubmedAbstract),
    "biological.tissueType":                 sum(1 for c in contexts if c.biological.tissueType),
    "biological.cellType":                   sum(1 for c in contexts if c.biological.cellType),
    "biological.sampleAttributes":           sum(1 for c in contexts if c.biological.sampleAttributes),
    "technical.libraryStrategy":             sum(1 for c in contexts if c.technical.libraryStrategy),
    "technical.libraryConstructionProtocol": sum(1 for c in contexts if c.technical.libraryConstructionProtocol),
}

pd.DataFrame({
    "field":     fields.keys(),
    "populated": fields.values(),
    "missing":   [n - v for v in fields.values()],
    "pct":       [f"{v / n * 100:.1f}%" for v in fields.values()],
})

,field,populated,missing,pct
0,studyDescription,801,0,100.0%
1,pubmedAbstract,530,271,66.2%
2,biological.tissueType,621,180,77.5%
3,biological.cellType,285,516,35.6%
4,biological.sampleAttributes,755,46,94.3%
5,technical.libraryStrategy,801,0,100.0%
6,technical.libraryConstructionProtocol,647,154,80.8%


## Warnings

In [16]:
# warnings
warned = [(ctx.accession, w) for ctx in contexts for w in ctx.warnings]

print(f"Records with warnings: {sum(1 for c in contexts if c.warnings)} / {n}")
print(f"Total warning entries: {len(warned)}\n")

if warned:
    warning_types = pd.Series([w for _, w in warned]).value_counts()
    print("Warning type breakdown:")
    print(warning_types.to_string())
    print("\nAffected accessions:")
    for acc, w in warned:
        print(f"  {acc}: {w}")

Records with warnings: 46 / 801
Total warning entries: 46

Warning type breakdown:
sample_xml_failed:HTTP GET failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN40627613': Client error '404 ' for url 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN40627613'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404    1
sample_xml_failed:HTTP GET failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166561': Client error '404 ' for url 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166561'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404    1
sample_xml_failed:HTTP GET failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166613': Client error '404 ' for url 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166613'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404    1
sample_xml_failed:HTTP GET failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN4616

## Distributions

In [17]:
# distributions
print("Library strategy:")
print(pd.Series([c.technical.libraryStrategy for c in contexts]).value_counts().to_string())

print("\nSpecies:")
print(pd.Series([c.biological.scientificName for c in contexts]).value_counts().to_string())

print("\nTissue type (top 20):")
print(pd.Series([c.biological.tissueType for c in contexts]).value_counts().head(20).to_string())

Library strategy:
RNA-Seq    770
OTHER       31

Species:
Homo sapiens    751
Mus musculus      4

Tissue type (top 20):
Lung                                                                                                                                                                                              167
lung                                                                                                                                                                                              142
Whole Lung Dissociate                                                                                                                                                                              49
More_fibrotic;Lung                                                                                                                                                                                 36
Less_fibrotic;Lung                                                                     

## Spot Checks

In [18]:
# spot checks
def print_ctx(ctx: ExperimentContext) -> None:
    print(f"Accession:         {ctx.accession}")
    print(f"Experiment title:  {ctx.experimentTitle}")
    print(f"Species:           {ctx.biological.scientificName}")
    print(f"Tissue:            {ctx.biological.tissueType}")
    print(f"Library strategy:  {ctx.technical.libraryStrategy}")
    print(f"Study description: {(ctx.study.studyDescription or '')[:200] if ctx.study else ''}")
    print(f"GEO accession:     {ctx.study.geoAccession if ctx.study else None}")
    print(f"PubMed abstract:   {(ctx.study.pubmedAbstract or '')[:200] if ctx.study else ''}")
    print(f"Warnings:          {ctx.warnings}")
    print()

print("First 3 records:")
for ctx in contexts[:3]:
    print_ctx(ctx)

no_abstract = [c for c in contexts if not (c.study and c.study.pubmedAbstract)]
print(f"Records without pubmedAbstract: {len(no_abstract)}")
print("\nFirst 5 without abstract:")
for ctx in no_abstract[:5]:
    print_ctx(ctx)

First 3 records:
Accession:         ERX11662371
Experiment title:  Illumina NovaSeq 6000 sequencing: Single cell RNA-seq atlas of human NSCLC lesions and non-involved tissue
Species:           Homo sapiens
Tissue:            None
Library strategy:  RNA-Seq
Study description: We performed single cell RNA sequencing (scRNA-seq) of NSCLC tumours and matched, adjacent, non-involved lung tissue from 24 patients. The data set is composed of approximately 900,000 cells from two 
GEO accession:     None
PubMed abstract:   
Warnings:          []

Accession:         SRX19233503
Experiment title:  NextSeq 500 sequencing: GSM7016385: Replicate 3, scRNAseq Homo sapiens RNA-Seq
Species:           Homo sapiens
Tissue:            Distal Thrombus removed from pulmonary artery of CTEPH patient
Library strategy:  RNA-Seq
Study description: Our current understanding of CTEPH pathobiology is primarily derived from cell-based studies limited by the use of specific cell markers or phenotypic modulation in ce

## Lookup Helpers

In [19]:
# lookup helpers
def get_context(accession: str) -> ExperimentContext | None:
    return next((ctx for ctx in contexts if ctx.accession == accession), None)


def experiment_context_summary(accession: str) -> str:
    ctx = get_context(accession)
    if not ctx:
        return f"No context found for accession: {accession}"

    parts = []
    if ctx.biological.tissueType:
        parts.append(f"Tissue: {ctx.biological.tissueType}")
    if ctx.technical.libraryStrategy:
        parts.append(f"Library strategy: {ctx.technical.libraryStrategy}")
    if ctx.technical.libraryConstructionProtocol:
        parts.append(f"Library prep: {ctx.technical.libraryConstructionProtocol}")
    if ctx.study and ctx.study.studyDescription:
        parts.append(f"Description: {ctx.study.studyDescription}")
    if ctx.study and ctx.study.pubmedAbstract:
        parts.append(f"Abstract: {ctx.study.pubmedAbstract}")

    return ". ".join(parts) if parts else "No descriptive fields found."


# example
print(experiment_context_summary("SRX17412841"))

Tissue: Whole Lung Dissociate. Library strategy: RNA-Seq. Library prep: Thawed Cells were rinsed with 10% FBS passed through 40 uM strainer 10X Genomics 3 prime v2. Description: Single Cell RNAseq of Whole Lung Dissociates from IPF, COPD and control lungs Overall design: Exploratory analysis of disease and healthy lung cell populations. Abstract: Airway mucociliary regeneration and function are key players for airway defense and are impaired in chronic obstructive pulmonary disease (COPD). Using transcriptome analysis in COPD-derived bronchial biopsies, we observed a positive correlation between cilia-related genes and microRNA-449 (miR449). In vitro, miR449 was strongly increased during airway epithelial mucociliary differentiation. In vivo, miR449 was upregulated during recovery from chemical or infective insults. miR0449-/- mice (both alleles are deleted) showed impaired ciliated epithelial regeneration after naphthalene and Haemophilus influenzae exposure, accompanied by more inten

## Text Length

In [20]:
# combined text length (study description + pubmed abstract)
CHAR_LIMIT = 300

combined_lengths = [
    len(ctx.study.studyDescription or "") + len(ctx.study.pubmedAbstract or "")
    if ctx.study else 0
    for ctx in contexts
]
below = sum(l < CHAR_LIMIT for l in combined_lengths)
print(f"{below} of {n} records ({below / n * 100:.1f}%) have combined studyDescription + pubmedAbstract < {CHAR_LIMIT} chars")

94 of 801 records (11.7%) have combined studyDescription + pubmedAbstract < 300 chars
